# LeetCode 505 — The Maze II bằng Python

Link đề: https://leetcode.com/problems/the-maze-ii/ · Độ khó: **Medium** · Chủ đề: Graph, Shortest Path, Dijkstra, BFS, DFS, Heap (Priority Queue), A* Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        ...
```

Ghi chú: đây là bài **Premium**. Đề và các ví dụ trong notebook được viết theo hiểu biết về bài; mọi đáp án đã được kiểm chứng bằng code (đối chiếu 5 cách làm độc lập và một bộ vét cạn), nhưng nếu bạn có tài khoản Premium thì nên đối chiếu lại đề gốc từng chữ. Ví dụ 3 viết lại theo trí nhớ nên cũng cần đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: quả bóng **lăn một mạch cho tới khi chạm tường** mới dừng được; ta cần **tổng số ô** ngắn nhất để bóng **dừng đúng tại đích** (đi ngang qua đích mà không dừng thì chưa tính).
2. Nắm các công cụ Python cần dùng: ma trận `list[list[int]]`, danh sách 4 hướng `(dr, dc)`, vòng `while` để lăn, `collections.deque`, `heapq` (hàng đợi ưu tiên), `math.inf`, tuple so sánh được trong heap.
3. Đi từ **vét cạn mọi đường đi** → **hàng đợi nới lỏng (SPFA)** → **Dijkstra** → **A\*** → **BFS trên trạng thái (ô, hướng) với bước đơn vị**; hiểu vì sao BFS "thường" cho sai đáp án bài này.
4. Hiểu mấu chốt: **đỉnh của đồ thị là các ô mà bóng có thể dừng**, **cạnh là một cú lăn trọn vẹn** với trọng số bằng số ô đã đi → đây là bài đường đi ngắn nhất có **trọng số dương** → Dijkstra.
5. Tránh các lỗi kinh điển: đếm số lần đổi hướng thay vì số ô, dừng bóng ngay khi đi ngang đích, đánh dấu "đã thăm" quá sớm khi trọng số không đều, lăn lố vào tường (quên lùi 1 bước).

## 2. Đề bài & ví dụ

Trong một mê cung (`maze`) có các ô trống (số `0`) và các bức tường (số `1`). Quả bóng có thể lăn **lên, xuống, trái, phải**, nhưng một khi đã lăn theo một hướng thì **không dừng lại cho đến khi chạm tường** (hoặc biên của mê cung). Khi bóng dừng, nó mới được chọn hướng lăn tiếp theo.

Cho mê cung `m x n`, vị trí xuất phát `start = [hàng, cột]` và đích `destination = [hàng, cột]`, hãy trả về **quãng đường ngắn nhất** để bóng **dừng tại đích**. Nếu không thể, trả về `-1`.

**Quãng đường** = số **ô trống** bóng đã đi qua, tính từ ô xuất phát (không tính) đến ô cuối (có tính).

Có thể coi biên của mê cung đều là tường.

**Ví dụ 1**

- Input: `maze = [[0,0,1,0,0],[0,0,0,0,0],[0,0,0,1,0],[1,1,0,1,1],[0,0,0,0,0]]`, `start = [0,4]`, `destination = [4,4]`
- Output: `12`
- Giải thích: một đường đi: trái -> xuống -> trái -> xuống -> phải -> xuống -> phải. Độ dài: `1 + 1 + 3 + 1 + 2 + 2 + 2 = 12`.

**Ví dụ 2**

- Input: cùng mê cung, `start = [0,4]`, `destination = [3,2]`
- Output: `-1`
- Giải thích: bóng có thể đi ngang qua ô `[3,2]` nhưng **không thể dừng** ở đó.

**Ví dụ 3**

- Input: `maze = [[0,0,0,0,0],[1,1,0,0,1],[0,0,0,0,0],[0,1,0,0,1],[0,1,0,0,0]]`, `start = [4,3]`, `destination = [0,1]`
- Output: `-1`

**Ràng buộc:**

- `m == maze.length`, `n == maze[i].length`, `1 <= m, n <= 100`
- `maze[i][j]` là `0` hoặc `1`.
- `start`, `destination` đều nằm trong mê cung và là ô trống; hai vị trí này **không trùng nhau** lúc đầu.
- Mê cung có **ít nhất 2 ô trống**.

In [1]:
from typing import List, Optional
from collections import deque
import heapq
import math
import random

maze1 = [[0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0],
         [1, 1, 0, 1, 1],
         [0, 0, 0, 0, 0]]
maze3 = [[0, 0, 0, 0, 0],
         [1, 1, 0, 0, 1],
         [0, 0, 0, 0, 0],
         [0, 1, 0, 0, 1],
         [0, 1, 0, 0, 0]]

examples = [
    (maze1, [0, 4], [4, 4], 12),
    (maze1, [0, 4], [3, 2], -1),
    (maze3, [4, 3], [0, 1], -1),
]

# In mê cung cho dễ nhìn: '#' là tường, '.' là ô trống, S là start, D là đích
def show(maze, start=None, dest=None):
    for r, row in enumerate(maze):
        line = ""
        for c, v in enumerate(row):
            if start is not None and [r, c] == list(start):
                line += "S "
            elif dest is not None and [r, c] == list(dest):
                line += "D "
            else:
                line += ("# " if v == 1 else ". ")
        print(line)

show(maze1, [0, 4], [4, 4])
print("Số ví dụ:", len(examples))

. . # . S 
. . . . . 
. . . # . 
# # . # # 
. . . . D 
Số ví dụ: 3


## 3. Kiến thức nền

### 3.1. Ma trận là "list của list", chỉ số `(hàng, cột)`

`maze[r][c]` là ô ở hàng `r`, cột `c`. Số hàng là `len(maze)`, số cột là `len(maze[0])`. Để biết một tọa độ có nằm trong mê cung không ta kiểm tra `0 <= r < m` và `0 <= c < n`. Trong bài này **ra khỏi biên cũng giống như đụng tường**.

In [2]:
maze = maze1
m, n = len(maze), len(maze[0])
print("m =", m, "n =", n)
print("maze[1][3] =", maze[1][3], "(ô trống)")
print("maze[2][3] =", maze[2][3], "(tường)")

def in_bounds(r, c):
    return 0 <= r < m and 0 <= c < n

print(in_bounds(0, 0), in_bounds(-1, 0), in_bounds(4, 5))

m = 5 n = 5
maze[1][3] = 0 (ô trống)
maze[2][3] = 1 (tường)
True False False


### 3.2. Bốn hướng đi bằng một danh sách `(dr, dc)`

Thay vì viết 4 khối `if` gần giống nhau, ta lưu 4 hướng thành các cặp "độ lệch hàng, độ lệch cột":

- lên: `(-1, 0)`, xuống: `(1, 0)`, trái: `(0, -1)`, phải: `(0, 1)`.

Cú pháp `for dr, dc in DIRS:` tự **tách (unpack)** mỗi cặp thành hai biến.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
NAMES = ["lên", "xuống", "trái", "phải"]

r, c = 1, 1
for (dr, dc), name in zip(DIRS, NAMES):
    print(f"từ ({r},{c}) đi {name}: ô kế tiếp là ({r + dr},{c + dc})")

từ (1,1) đi lên: ô kế tiếp là (0,1)
từ (1,1) đi xuống: ô kế tiếp là (2,1)
từ (1,1) đi trái: ô kế tiếp là (1,0)
từ (1,1) đi phải: ô kế tiếp là (1,2)


### 3.3. "Lăn đến khi chạm tường" = vòng `while`

Đây là điểm khác biệt chính so với bài mê cung thông thường. Từ ô `(r, c)` theo hướng `(dr, dc)`:

1. Nhìn ô **kế tiếp** `(r+dr, c+dc)`.
2. Nếu nó nằm trong mê cung **và** là ô trống thì bước tới, cộng 1 vào số bước.
3. Lặp lại cho tới khi ô kế tiếp là tường/ngoài biên. Lúc đó bóng đang đứng ở ô **cuối cùng còn trống** — đó là chỗ bóng dừng.

Chú ý: ta kiểm tra ô **kế tiếp** trước khi bước, nên không bao giờ "lố" vào tường và không cần lùi lại.

In [4]:
def roll(maze, r, c, dr, dc):
    """Lăn từ (r, c) theo hướng (dr, dc). Trả về (hàng dừng, cột dừng, số ô đã đi)."""
    m, n = len(maze), len(maze[0])
    steps = 0
    while 0 <= r + dr < m and 0 <= c + dc < n and maze[r + dr][c + dc] == 0:
        r += dr
        c += dc
        steps += 1
    return r, c, steps

# từ start = (0, 4) của ví dụ 1
for (dr, dc), name in zip(DIRS, NAMES):
    print(name, "->", roll(maze1, 0, 4, dr, dc))

lên -> (0, 4, 0)
xuống -> (2, 4, 2)
trái -> (0, 3, 1)
phải -> (0, 4, 0)


Kết quả cho thấy: lăn **trái** từ `(0,4)` đi 1 ô và dừng ở `(0,3)` (vì `(0,2)` là tường). Lăn **xuống** đi 2 ô, dừng ở `(2,4)` (vì `(3,4)` là tường). Lên và phải: bóng đã sát biên nên đi `0` ô.

Một cú lăn đi `0` ô nghĩa là "không có tác dụng" — ta sẽ bỏ qua nó (`steps == 0`).

### 3.4. `deque` và `heapq`

- `collections.deque`: hàng đợi hai đầu. `popleft()` lấy phần tử vào **trước nhất** — dùng cho BFS.
- `heapq`: **min-heap**. `heappush(h, x)` thêm, `heappop(h)` lấy phần tử **nhỏ nhất**. Với tuple, Python so sánh **từng phần tử từ trái sang phải**, nên `(khoảng_cách, hàng, cột)` sẽ lấy ra phần tử có khoảng cách nhỏ nhất trước.

In [5]:
q = deque([1, 2, 3])
q.append(4)
print("popleft:", q.popleft(), "còn", list(q))

h = []
for item in [(5, 0, 1), (2, 3, 3), (9, 0, 0), (2, 1, 1)]:
    heapq.heappush(h, item)
out = []
while h:
    out.append(heapq.heappop(h))
print(out)   # tăng dần theo phần tử đầu, hòa thì xét tiếp phần tử sau

popleft: 1 còn [2, 3, 4]
[(2, 1, 1), (2, 3, 3), (5, 0, 1), (9, 0, 0)]


### 3.5. `math.inf` và bảng khoảng cách

Ta cần một bảng `dist[r][c]` ghi **quãng đường ngắn nhất đã biết** để tới `(r, c)`. Ban đầu chưa biết gì nên điền `math.inf` ("vô cực", lớn hơn mọi số). Mọi so sánh `x < math.inf` đều đúng với số hữu hạn `x`, và `math.inf + 5` vẫn là `inf`.

In [6]:
INF = math.inf
dist = [[INF] * 5 for _ in range(3)]      # đừng viết [[INF]*5]*3 (3 hàng sẽ là CÙNG MỘT list)
dist[0][0] = 0
print(dist)
print(7 < INF, INF + 5, min(INF, 12))

bad = [[INF] * 5] * 3
bad[0][0] = 0
print("Sai lầm [[...]*5]*3:", bad)

[[0, inf, inf, inf, inf], [inf, inf, inf, inf, inf], [inf, inf, inf, inf, inf]]
True inf 12
Sai lầm [[...]*5]*3: [[0, inf, inf, inf, inf], [0, inf, inf, inf, inf], [0, inf, inf, inf, inf]]


### 3.6. "Nới lỏng cạnh" (relaxation)

Ý tưởng cốt lõi của mọi thuật toán đường đi ngắn nhất: nếu đã đứng ở `u` với khoảng cách `dist[u]`, và có cạnh `u -> v` nặng `w`, thì ta **có một đường tới `v`** dài `dist[u] + w`. Nếu nó **ngắn hơn** `dist[v]` hiện có thì cập nhật:

```python
if dist[u] + w < dist[v]:
    dist[v] = dist[u] + w
```

Các thuật toán khác nhau (SPFA, Dijkstra, A*) chỉ khác nhau ở **thứ tự** xét các đỉnh `u`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tính tay ví dụ 1 từ `S = (0,4)`.

```
. . # . S      hàng 0
. . . . .      hàng 1
. . . # .      hàng 2
# # . # #      hàng 3
. . . . D      hàng 4
```

**Từ `(0,4)`**: trái dừng ở `(0,3)` (đi 1), xuống dừng ở `(2,4)` (đi 2). Lên/phải không đi được.

**Từ `(0,3)`** (khoảng cách 1): xuống `(1,3)` rồi `(2,3)` là tường nên dừng ở `(1,3)`, tổng `1 + 1 = 2`. Trái bị tường `(0,2)` chặn. Phải quay lại `(0,4)`: tổng 2 (không có ích).

**Từ `(2,4)`** (khoảng cách 2): lên về `(0,4)`, trái: `(2,3)` là tường, không đi. Xuống: `(3,4)` là tường.

Quan sát quan trọng: **mỗi cú lăn có độ dài khác nhau** (1, 2, 3, ...). Vì vậy "đi qua ít cú lăn" **không** có nghĩa là "đi ít ô". Ta phải cộng độ dài thật.

Còn một điều nữa: **đỉnh của bài toán là ô mà bóng dừng được**, không phải mọi ô trống. Ở ví dụ 2, ô `(3,2)` là ô trống nhưng không có cú lăn nào kết thúc ở đó, nên đáp án là `-1`.

Hãy để máy liệt kê "các điểm dừng kế tiếp" từ mỗi ô để thấy rõ đồ thị ẩn:

In [7]:
def next_stops(maze, r, c):
    """Các (ô_dừng, độ_dài) mà bóng có thể tới bằng đúng MỘT cú lăn từ (r, c)."""
    res = []
    for dr, dc in DIRS:
        nr, nc, steps = roll(maze, r, c, dr, dc)
        if steps > 0:
            res.append(((nr, nc), steps))
    return res

for cell in [(0, 4), (0, 3), (2, 4), (1, 3)]:
    print(cell, "->", next_stops(maze1, *cell))

(0, 4) -> [((2, 4), 2), ((0, 3), 1)]
(0, 3) -> [((1, 3), 1), ((0, 4), 1)]
(2, 4) -> [((0, 4), 2)]
(1, 3) -> [((0, 3), 1), ((1, 0), 3), ((1, 4), 1)]


Đây chính là một **đồ thị có hướng có trọng số**: đỉnh là ô dừng, cạnh là cú lăn, trọng số là số ô. Việc còn lại là tìm đường đi ngắn nhất từ `start` tới `destination` trên đồ thị này.

## 5. Cách 1 — Brute Force / baseline: vét cạn mọi đường đi (DFS)

**Ý tưởng:** thử **mọi dãy cú lăn** có thể, miễn là không đi vào lại một ô dừng đã nằm trên đường đi hiện tại (để không lặp vô hạn). Mỗi lần tới được `destination` (bóng dừng ở đó) thì cập nhật đáp án nhỏ nhất.

Đây là DFS "liệt kê mọi đường đi đơn" (simple path) + quay lui (backtracking): đánh dấu ô khi bước vào, **bỏ đánh dấu** khi quay ra.

In [8]:
def shortest_brute(maze, start, destination):
    best = math.inf
    on_path = {tuple(start)}                  # các ô dừng đang nằm trên đường đi hiện tại

    def dfs(r, c, total):
        nonlocal best
        if [r, c] == list(destination):       # bóng ĐANG DỪNG ở đích
            best = min(best, total)
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and (nr, nc) not in on_path:
                on_path.add((nr, nc))
                dfs(nr, nc, total + steps)
                on_path.discard((nr, nc))     # quay lui

    dfs(start[0], start[1], 0)
    return -1 if best == math.inf else best

for mz, s, d, exp in examples:
    print(shortest_brute(mz, s, d), "kỳ vọng", exp)

12 kỳ vọng 12
-1 kỳ vọng -1
-1 kỳ vọng -1


**Độ phức tạp.** Số đường đi đơn trong đồ thị có thể **tăng theo hàm mũ** theo số ô dừng: Time `O(V!)`-ish trong trường hợp xấu (với `V` là số ô dừng, `V ≤ m·n`), Space `O(V)` cho ngăn xếp đệ quy.

**Nút thắt:** ta xét đi xét lại cùng một ô dừng bằng nhiều đường khác nhau, dù ta chỉ quan tâm *đường ngắn nhất tới ô đó*. Nếu đã biết "tới ô X nhanh nhất mất 7", thì mọi đường khác tới X dài hơn 7 đều vô ích. Đó là ý tưởng lưu `dist[]`.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Đồ thị ẩn:** đỉnh = ô dừng được; cạnh = một cú lăn; trọng số = số ô (luôn `>= 1`, dương).
2. **Đường đi ngắn nhất có trọng số dương** → có thể dùng **Dijkstra**: luôn xử lý ô có `dist` nhỏ nhất chưa xử lý; khi một ô được lấy ra khỏi heap thì `dist` của nó đã **chốt**.
3. Cũng có thể dùng **hàng đợi + nới lỏng** (SPFA / "BFS cho phép thăm lại"): mỗi khi `dist[v]` giảm thì đưa `v` vào hàng đợi lần nữa. Đúng nhưng có thể chậm hơn.
4. Vì khoảng cách thật ≥ khoảng cách Manhattan, ta có thể dùng **A\*** với heuristic Manhattan để heap đi "hướng về đích".
5. Hoặc đổi góc nhìn: chia mỗi cú lăn thành **các bước đơn vị**, nhớ thêm **hướng đang lăn** trong trạng thái `(r, c, d)`. Khi đó mọi cạnh nặng `1`, nên BFS thường là đủ.

**Hai bẫy cần nhớ cho mọi cách:**

- Phải **dừng** ở đích mới tính (đích phải là điểm dừng — với Dijkstra, đích được cập nhật `dist` chỉ khi một cú lăn kết thúc ở đó, nên tự động đúng).
- Không dùng BFS "mỗi cú lăn là 1 bước": nó tối thiểu hóa **số cú lăn**, không phải **số ô**.

## 7. Các cách tiếp theo

### Cách 2 — Hàng đợi + nới lỏng (SPFA, "BFS thăm lại")

**Giải thích từng đoạn:**

1. `dist` toàn `inf`, trừ `dist[start] = 0`. Đưa `start` vào `deque`.
2. Lấy ô `(r, c)` ở đầu hàng đợi; với mỗi hướng, lăn tới `(nr, nc)` với `steps` ô.
3. **Nới lỏng:** nếu `dist[r][c] + steps < dist[nr][nc]` thì cập nhật và đưa `(nr, nc)` vào hàng đợi (vì các ô sau nó có thể được cải thiện).
4. Kết thúc khi hàng đợi rỗng; đáp án là `dist[destination]` (hoặc `-1` nếu còn `inf`).

In [9]:
def shortest_spfa(maze, start, destination):
    m, n = len(maze), len(maze[0])
    dist = [[math.inf] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    q = deque([(start[0], start[1])])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and dist[r][c] + steps < dist[nr][nc]:
                dist[nr][nc] = dist[r][c] + steps
                q.append((nr, nc))
    ans = dist[destination[0]][destination[1]]
    return -1 if ans == math.inf else ans

for mz, s, d, exp in examples:
    print(shortest_spfa(mz, s, d), "kỳ vọng", exp)

12 kỳ vọng 12
-1 kỳ vọng -1
-1 kỳ vọng -1


In [10]:
class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        dist = [[float('inf')] * n for _ in range(m)]
        dist[start[0]][start[1]] = 0
        q = deque([(start[0], start[1])])
        while q:
            r, c = q.popleft()
            for dr, dc in dirs:
                nr, nc, steps = r, c, 0
                while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                    nr += dr
                    nc += dc
                    steps += 1
                if steps and dist[r][c] + steps < dist[nr][nc]:
                    dist[nr][nc] = dist[r][c] + steps
                    q.append((nr, nc))
        ans = dist[destination[0]][destination[1]]
        return -1 if ans == float('inf') else ans

print([Solution().shortestDistance(mz, s, d) for mz, s, d, _ in examples])

[12, -1, -1]


**Độ phức tạp (cách 2):** mỗi cú lăn tốn `O(max(m, n))`. Số lần một ô bị đưa vào hàng đợi lại phụ thuộc dữ liệu; trường hợp xấu cỡ `O(V · E)` lần nới lỏng. Thực tế chạy nhanh trên mê cung nhỏ, nhưng không có bảo đảm tốt như Dijkstra. Space `O(m · n)`.

### Cách 3 — Dijkstra với heap (cách chuẩn)

**Giải thích từng đoạn:**

1. Heap chứa `(khoảng_cách, hàng, cột)`; ban đầu `(0, sr, sc)`.
2. `heappop` lấy ô có khoảng cách **nhỏ nhất**. Nếu khoảng cách trong heap **lớn hơn** `dist` đã ghi (bản cũ, "lỗi thời") thì bỏ qua.
3. Nếu ô này là đích → **return** ngay, vì Dijkstra đảm bảo khoảng cách lấy ra đầu tiên là nhỏ nhất.
4. Với mỗi hướng, lăn tới ô dừng, nới lỏng và `heappush` nếu cải thiện.
5. Heap rỗng mà chưa gặp đích → `-1`.

Vì sao return sớm là an toàn? Mọi trọng số đều dương, nên các ô còn trong heap có khoảng cách `>=` ô vừa lấy ra, và chúng không thể đem lại đường ngắn hơn tới ô vừa lấy ra.

In [11]:
def shortest_dijkstra(maze, start, destination):
    m, n = len(maze), len(maze[0])
    dist = [[math.inf] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    heap = [(0, start[0], start[1])]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[r][c]:                       # bản cũ, bỏ qua
            continue
        if [r, c] == list(destination):          # lần đầu lấy ra đích = ngắn nhất
            return d
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and d + steps < dist[nr][nc]:
                dist[nr][nc] = d + steps
                heapq.heappush(heap, (d + steps, nr, nc))
    return -1

for mz, s, d, exp in examples:
    print(shortest_dijkstra(mz, s, d), "kỳ vọng", exp)

12 kỳ vọng 12
-1 kỳ vọng -1
-1 kỳ vọng -1


In [12]:
class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        dist = [[float('inf')] * n for _ in range(m)]
        dist[start[0]][start[1]] = 0
        heap = [(0, start[0], start[1])]
        while heap:
            d, r, c = heapq.heappop(heap)
            if d > dist[r][c]:
                continue
            if r == destination[0] and c == destination[1]:
                return d
            for dr, dc in dirs:
                nr, nc, steps = r, c, 0
                while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                    nr += dr
                    nc += dc
                    steps += 1
                if steps and d + steps < dist[nr][nc]:
                    dist[nr][nc] = d + steps
                    heapq.heappush(heap, (d + steps, nr, nc))
        return -1

print([Solution().shortestDistance(mz, s, d) for mz, s, d, _ in examples])

[12, -1, -1]


**Độ phức tạp (cách 3):** có tối đa `V ≤ m·n` ô dừng; mỗi ô được "xử lý thật" một lần, mỗi lần thử 4 hướng, mỗi cú lăn `O(max(m,n))`. Số lần `heappush` tối đa bằng số lần nới lỏng thành công `O(4·m·n)`, mỗi lần `O(log(m·n))`.

- **Time:** `O(m·n·(m+n) + m·n·log(m·n))`, cụ thể có thể viết `O(m·n·max(m,n))` khi bỏ phần log.
- **Space:** `O(m·n)` cho `dist` và heap.

### Cách 4 — A\* (Dijkstra có "la bàn")

**Ý tưởng:** thay vì sắp heap theo `g` (quãng đường đã đi), sắp theo `f = g + h`, với `h(ô)` = **khoảng cách Manhattan** `|r - dr| + |c - dc|` tới đích.

Vì sao dùng được `h` này?

- **Không đánh giá quá cao (admissible):** bóng lăn theo các đoạn thẳng, mỗi ô đi qua dịch chuyển đúng 1 đơn vị theo hàng hoặc cột, nên số ô đi **không bao giờ ít hơn** khoảng cách Manhattan.
- **Nhất quán (consistent):** một cú lăn thẳng từ `u` sang `v` dài đúng `|u - v|_Manhattan`, và Manhattan thỏa bất đẳng thức tam giác, nên `h(u) <= w(u, v) + h(v)`.

Do đó khi lấy được đích ra khỏi heap lần đầu thì `g` của nó là nhỏ nhất. Ta vẫn giữ kiểm tra `d > dist[r][c]` để bỏ bản cũ.

In [13]:
def shortest_astar(maze, start, destination):
    m, n = len(maze), len(maze[0])
    dr_, dc_ = destination
    def h(r, c):
        return abs(r - dr_) + abs(c - dc_)

    dist = [[math.inf] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    heap = [(h(*start), 0, start[0], start[1])]      # (f, g, r, c)
    while heap:
        f, g, r, c = heapq.heappop(heap)
        if g > dist[r][c]:
            continue
        if [r, c] == list(destination):
            return g
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and g + steps < dist[nr][nc]:
                dist[nr][nc] = g + steps
                heapq.heappush(heap, (g + steps + h(nr, nc), g + steps, nr, nc))
    return -1

for mz, s, d, exp in examples:
    print(shortest_astar(mz, s, d), "kỳ vọng", exp)

12 kỳ vọng 12
-1 kỳ vọng -1
-1 kỳ vọng -1


In [14]:
class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        tr, tc = destination
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        dist = [[float('inf')] * n for _ in range(m)]
        dist[start[0]][start[1]] = 0
        heap = [(abs(start[0] - tr) + abs(start[1] - tc), 0, start[0], start[1])]
        while heap:
            f, g, r, c = heapq.heappop(heap)
            if g > dist[r][c]:
                continue
            if r == tr and c == tc:
                return g
            for dr, dc in dirs:
                nr, nc, steps = r, c, 0
                while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                    nr += dr
                    nc += dc
                    steps += 1
                if steps and g + steps < dist[nr][nc]:
                    dist[nr][nc] = g + steps
                    heapq.heappush(heap, (g + steps + abs(nr - tr) + abs(nc - tc), g + steps, nr, nc))
        return -1

print([Solution().shortestDistance(mz, s, d) for mz, s, d, _ in examples])

[12, -1, -1]


**Độ phức tạp (cách 4):** trường hợp xấu giống Dijkstra: `O(m·n·(m+n) + m·n·log(m·n))`, Space `O(m·n)`. Trong thực tế A* thường mở rộng ít ô hơn khi đích ở xa theo một hướng rõ ràng.

### Cách 5 — BFS trên trạng thái `(hàng, cột, hướng)` với bước đơn vị

**Ý tưởng khác hẳn:** thay vì coi cả cú lăn là một cạnh, ta cho bóng **đi từng ô một** (cạnh nặng `1`) và **nhớ hướng** đang lăn:

- Trạng thái `(r, c, d)`: bóng đang ở ô `(r, c)`, vừa đi vào ô này theo hướng `d`. Trạng thái xuất phát `(sr, sc, 4)` với `d = 4` nghĩa là "đang đứng yên, chọn hướng nào cũng được".
- Nếu ô kế tiếp theo hướng `d` **còn trống** → **bắt buộc** đi tiếp: `(r+dr, c+dc, d)` tốn 1.
- Nếu ô kế tiếp theo hướng `d` là **tường** → bóng **vừa dừng**. Nếu `(r, c)` là đích thì ta có đáp án. Nếu không, bóng được đổi sang hướng `d2 ≠ d` bất kỳ còn đi được: `(r+dr2, c+dc2, d2)` tốn 1.

Vì mọi cạnh đều nặng `1`, **BFS thường** cho khoảng cách ngắn nhất, và lần đầu BFS gặp "đích + vừa dừng" là đáp án. Số trạng thái ≤ `5·m·n`.

In [15]:
def shortest_bfs_dir(maze, start, destination):
    m, n = len(maze), len(maze[0])
    def free(r, c):
        return 0 <= r < m and 0 <= c < n and maze[r][c] == 0

    s0 = (start[0], start[1], 4)                 # 4 = chưa có hướng
    dist = {s0: 0}
    q = deque([s0])
    while q:
        r, c, d = q.popleft()
        cur = dist[(r, c, d)]
        if d < 4:
            nr, nc = r + DIRS[d][0], c + DIRS[d][1]
            if free(nr, nc):                     # chưa chạm tường: bắt buộc đi tiếp
                nxt = (nr, nc, d)
                if nxt not in dist:
                    dist[nxt] = cur + 1
                    q.append(nxt)
                continue
            if [r, c] == list(destination):      # vừa dừng ở đích
                return cur
        for d2, (dr, dc) in enumerate(DIRS):     # đang đứng yên: chọn hướng mới
            if d2 == d:
                continue
            nr, nc = r + dr, c + dc
            if free(nr, nc):
                nxt = (nr, nc, d2)
                if nxt not in dist:
                    dist[nxt] = cur + 1
                    q.append(nxt)
    return -1

for mz, s, d, exp in examples:
    print(shortest_bfs_dir(mz, s, d), "kỳ vọng", exp)

12 kỳ vọng 12
-1 kỳ vọng -1
-1 kỳ vọng -1


In [16]:
class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))

        def free(r, c):
            return 0 <= r < m and 0 <= c < n and maze[r][c] == 0

        s0 = (start[0], start[1], 4)
        dist = {s0: 0}
        q = deque([s0])
        while q:
            r, c, d = q.popleft()
            cur = dist[(r, c, d)]
            if d < 4:
                nr, nc = r + dirs[d][0], c + dirs[d][1]
                if free(nr, nc):
                    nxt = (nr, nc, d)
                    if nxt not in dist:
                        dist[nxt] = cur + 1
                        q.append(nxt)
                    continue
                if r == destination[0] and c == destination[1]:
                    return cur
            for d2 in range(4):
                if d2 == d:
                    continue
                nr, nc = r + dirs[d2][0], c + dirs[d2][1]
                if free(nr, nc):
                    nxt = (nr, nc, d2)
                    if nxt not in dist:
                        dist[nxt] = cur + 1
                        q.append(nxt)
        return -1

print([Solution().shortestDistance(mz, s, d) for mz, s, d, _ in examples])

[12, -1, -1]


**Độ phức tạp (cách 5):** có tối đa `5·m·n` trạng thái, mỗi trạng thái xử lý `O(4)`.

- **Time:** `O(m·n)` (hằng số cỡ 20).
- **Space:** `O(m·n)`.

Đây là cách **nhanh nhất về lý thuyết** (không có `log`, không có thêm hệ số `max(m,n)`), đổi lại phải nhớ hướng trong trạng thái và cẩn thận phần "vừa dừng". Với mê cung `100 x 100`, cách nào trong 3, 4, 5 cũng chạy rất nhanh.

## 8. TRACE từng bước

Ta trace **Dijkstra** trên ví dụ 1 (`start = (0,4)`, `destination = (4,4)`). Mỗi dòng in một lần `heappop`: giá trị lấy ra, trạng thái heap, và các nới lỏng xảy ra.

Bảng trạng thái cần theo dõi:

| Cột | Ý nghĩa |
|---|---|
| `pop` | `(d, r, c)` vừa lấy khỏi heap |
| `bỏ qua?` | `True` nếu `d > dist[r][c]` (bản cũ) |
| `lăn` | hướng, ô dừng, số ô, và `dist` mới nếu được cải thiện |
| `heap` | nội dung heap sau khi xử lý |

In [17]:
def trace_dijkstra(maze, start, destination):
    m, n = len(maze), len(maze[0])
    dist = [[math.inf] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    heap = [(0, start[0], start[1])]
    step = 0
    while heap:
        d, r, c = heapq.heappop(heap)
        step += 1
        print(f"--- bước {step}: pop (d={d}, r={r}, c={c})")
        if d > dist[r][c]:
            print("    bản cũ, bỏ qua")
            continue
        if [r, c] == list(destination):
            print("    ĐÃ TỚI ĐÍCH, trả về", d)
            return d
        for (dr, dc), name in zip(DIRS, NAMES):
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps == 0:
                continue
            if d + steps < dist[nr][nc]:
                old = dist[nr][nc]
                dist[nr][nc] = d + steps
                heapq.heappush(heap, (d + steps, nr, nc))
                print(f"    {name}: dừng ({nr},{nc}), {steps} ô, dist {old} -> {d + steps}  [push]")
            else:
                print(f"    {name}: dừng ({nr},{nc}), {steps} ô, d+steps={d + steps} không tốt hơn {dist[nr][nc]}")
        print("    heap:", sorted(heap))
    print("heap rỗng, không có đường")
    return -1

trace_dijkstra(maze1, [0, 4], [4, 4])

--- bước 1: pop (d=0, r=0, c=4)
    xuống: dừng (2,4), 2 ô, dist inf -> 2  [push]
    trái: dừng (0,3), 1 ô, dist inf -> 1  [push]
    heap: [(1, 0, 3), (2, 2, 4)]
--- bước 2: pop (d=1, r=0, c=3)
    xuống: dừng (1,3), 1 ô, dist inf -> 2  [push]
    phải: dừng (0,4), 1 ô, d+steps=2 không tốt hơn 0
    heap: [(2, 1, 3), (2, 2, 4)]
--- bước 3: pop (d=2, r=1, c=3)
    lên: dừng (0,3), 1 ô, d+steps=3 không tốt hơn 1
    trái: dừng (1,0), 3 ô, dist inf -> 5  [push]
    phải: dừng (1,4), 1 ô, dist inf -> 3  [push]
    heap: [(2, 2, 4), (3, 1, 4), (5, 1, 0)]
--- bước 4: pop (d=2, r=2, c=4)
    lên: dừng (0,4), 2 ô, d+steps=4 không tốt hơn 0
    heap: [(3, 1, 4), (5, 1, 0)]
--- bước 5: pop (d=3, r=1, c=4)
    lên: dừng (0,4), 1 ô, d+steps=4 không tốt hơn 0
    xuống: dừng (2,4), 1 ô, d+steps=4 không tốt hơn 2
    trái: dừng (1,0), 4 ô, d+steps=7 không tốt hơn 5
    heap: [(5, 1, 0)]
--- bước 6: pop (d=5, r=1, c=0)
    lên: dừng (0,0), 1 ô, dist inf -> 6  [push]
    xuống: dừng (2,0), 1 ô, dist

12

Nhìn vào trace: ô `(4,4)` được tới lần đầu từ `(4,2)` với `dist = 12`; về sau còn một đường khác từ `(4,0)` tới `(4,4)` nhưng dài `16` nên bị bỏ ("không tốt hơn"). Heap luôn lấy ra khoảng cách nhỏ nhất, nên khi `(4,4)` được pop, `12` là chắc chắn nhỏ nhất. (Trên ví dụ này không có ô nào bị cải thiện lần hai; trên mê cung khác điều đó xảy ra, và đó là lý do ta cần kiểm tra `d > dist[r][c]` để bỏ bản cũ.)

Bây giờ trace **ví dụ 2** (đích `(3,2)` không thể dừng): heap sẽ cạn mà không bao giờ pop `(3,2)`.

In [18]:
res = trace_dijkstra(maze1, [0, 4], [3, 2])
print("kết quả:", res)

--- bước 1: pop (d=0, r=0, c=4)
    xuống: dừng (2,4), 2 ô, dist inf -> 2  [push]
    trái: dừng (0,3), 1 ô, dist inf -> 1  [push]
    heap: [(1, 0, 3), (2, 2, 4)]
--- bước 2: pop (d=1, r=0, c=3)
    xuống: dừng (1,3), 1 ô, dist inf -> 2  [push]
    phải: dừng (0,4), 1 ô, d+steps=2 không tốt hơn 0
    heap: [(2, 1, 3), (2, 2, 4)]
--- bước 3: pop (d=2, r=1, c=3)
    lên: dừng (0,3), 1 ô, d+steps=3 không tốt hơn 1
    trái: dừng (1,0), 3 ô, dist inf -> 5  [push]
    phải: dừng (1,4), 1 ô, dist inf -> 3  [push]
    heap: [(2, 2, 4), (3, 1, 4), (5, 1, 0)]
--- bước 4: pop (d=2, r=2, c=4)
    lên: dừng (0,4), 2 ô, d+steps=4 không tốt hơn 0
    heap: [(3, 1, 4), (5, 1, 0)]
--- bước 5: pop (d=3, r=1, c=4)
    lên: dừng (0,4), 1 ô, d+steps=4 không tốt hơn 0
    xuống: dừng (2,4), 1 ô, d+steps=4 không tốt hơn 2
    trái: dừng (1,0), 4 ô, d+steps=7 không tốt hơn 5
    heap: [(5, 1, 0)]
--- bước 6: pop (d=5, r=1, c=0)
    lên: dừng (0,0), 1 ô, dist inf -> 6  [push]
    xuống: dừng (2,0), 1 ô, dist

## 9. Code nộp LeetCode

Dijkstra là cách chuẩn, dễ chứng minh đúng. Đây là code đúng class/method/signature của LeetCode (với `from typing import List` và `heapq` ở đầu file nếu chạy ngoài LeetCode; trên LeetCode hai thứ này đã có sẵn):

In [19]:
import heapq
from typing import List


class Solution:
    def shortestDistance(self, maze: List[List[int]], start: List[int], destination: List[int]) -> int:
        m, n = len(maze), len(maze[0])
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        dist = [[float('inf')] * n for _ in range(m)]
        dist[start[0]][start[1]] = 0
        heap = [(0, start[0], start[1])]
        while heap:
            d, r, c = heapq.heappop(heap)
            if d > dist[r][c]:
                continue
            if r == destination[0] and c == destination[1]:
                return d
            for dr, dc in dirs:
                nr, nc, steps = r, c, 0
                while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                    nr += dr
                    nc += dc
                    steps += 1
                if steps and d + steps < dist[nr][nc]:
                    dist[nr][nc] = d + steps
                    heapq.heappush(heap, (d + steps, nr, nc))
        return -1


for mz, s, d, exp in examples:
    got = Solution().shortestDistance(mz, s, d)
    print(got, exp)
    assert got == exp

12 12
-1 -1
-1 -1


## 10. Vì sao thuật toán đúng

**Mô hình hóa đúng.** Mỗi chuỗi hành động hợp lệ của bóng chính là một dãy các cú lăn; sau mỗi cú lăn bóng dừng ở một ô xác định duy nhất (do luật "lăn tới khi chạm tường"). Vì vậy tập đường đi hợp lệ tương ứng 1-1 với đường đi trong đồ thị (đỉnh = ô dừng, cạnh = cú lăn). Quãng đường bài hỏi là tổng trọng số cạnh.

**Dijkstra đúng.** Trọng số mọi cạnh `>= 1 > 0`. Khi một ô `u` được lấy ra khỏi heap với khoảng cách `d`, giả sử tồn tại đường ngắn hơn tới `u`. Đường đó phải đi qua một ô `x` đầu tiên chưa được chốt; ô `x` có `dist[x] <=` độ dài đoạn đầu của đường đó `< d` (vì trọng số dương), và `x` đang nằm trong heap, mâu thuẫn với việc `u` là phần tử nhỏ nhất. Vậy `d` là tối ưu.

**Điều kiện "phải dừng ở đích".** `dist[destination]` chỉ bị gán khi một cú lăn **kết thúc** ở đích, tức là bóng thực sự dừng ở đó. Đi ngang qua đích không cập nhật gì. Vì thế ví dụ 2 trả `-1`.

**Bản A\* đúng.** Heuristic nhất quán nên thứ tự lấy ra theo `f` vẫn bảo đảm `g` tối ưu tại lần lấy đầu tiên (ta vẫn bỏ qua bản cũ để an toàn).

**Bản BFS theo `(ô, hướng)` đúng.** Mỗi đường đi hợp lệ của bóng tương ứng 1-1 với một dãy bước đơn vị mà ta đã mô hình hóa (bắt buộc đi tiếp khi chưa chạm tường, chỉ đổi hướng khi vừa dừng). Mọi cạnh nặng `1` nên BFS cho đường ngắn nhất theo số bước, mà số bước = số ô.

## 11. Phân tích Time / Space Complexity

Gọi `V ≤ m·n` là số ô có thể là điểm dừng, `L = max(m, n)` là độ dài tối đa một cú lăn.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn DFS | cấp mũ theo `V` | `O(V)` |
| 2. SPFA (hàng đợi) | `O(V · E · L)` trường hợp xấu | `O(m·n)` |
| 3. Dijkstra | `O(m·n·L + m·n·log(m·n))` | `O(m·n)` |
| 4. A* | như Dijkstra (thực tế thường nhanh hơn) | `O(m·n)` |
| 5. BFS `(ô, hướng)` | `O(m·n)` | `O(m·n)` |

Với `m, n ≤ 100`: `m·n = 10^4`, `L = 100`, nên Dijkstra tốn cỡ `10^6` phép — rất nhẹ.

## 12. Bộ test

### 12.1. Ví dụ đề và edge case

Ta gom 5 cách chạy được vào một danh sách để gọi cùng lúc (trừ cách vét cạn, chỉ dùng cho mê cung nhỏ).

In [20]:
solvers = {
    "brute": shortest_brute,
    "spfa": shortest_spfa,
    "dijkstra": shortest_dijkstra,
    "astar": shortest_astar,
    "bfs_dir": shortest_bfs_dir,
}

for mz, s, d, exp in examples:
    for name, fn in solvers.items():
        assert fn(mz, s, d) == exp, (name, s, d)
    assert Solution().shortestDistance(mz, s, d) == exp
print("Ví dụ đề: OK cho cả 5 cách + code nộp")

Ví dụ đề: OK cho cả 5 cách + code nộp


In [21]:
edge_cases = [
    # (maze, start, dest, expected, mô tả)
    ([[0, 0]], [0, 0], [0, 1], 1, "hàng 1x2: lăn phải 1 ô"),
    ([[0, 0, 0]], [0, 0], [0, 2], 2, "hàng 1x3: lăn tới cuối"),
    ([[0, 0, 0]], [0, 0], [0, 1], -1, "ô giữa chỉ đi ngang qua, không dừng được"),
    ([[0], [0]], [0, 0], [1, 0], 1, "cột 2x1"),
    ([[0, 1, 0]], [0, 0], [0, 2], -1, "bị tường chắn"),
    ([[0, 0], [0, 0]], [0, 0], [1, 1], 2, "2x2 trống: phải rồi xuống"),
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], [0, 0], [2, 2], 4, "vòng quanh khối tường giữa"),
]
for mz, s, d, exp, note in edge_cases:
    for name, fn in solvers.items():
        got = fn(mz, s, d)
        assert got == exp, (name, note, got, exp)
    assert Solution().shortestDistance(mz, s, d) == exp
    print("OK:", note, "->", exp)

OK: hàng 1x2: lăn phải 1 ô -> 1
OK: hàng 1x3: lăn tới cuối -> 2
OK: ô giữa chỉ đi ngang qua, không dừng được -> -1
OK: cột 2x1 -> 1
OK: bị tường chắn -> -1
OK: 2x2 trống: phải rồi xuống -> 2
OK: vòng quanh khối tường giữa -> 4


### 12.2. Phản ví dụ cho các cách làm sai

**Sai 1 — BFS mỗi cú lăn là 1 bước** (tối thiểu hóa số cú lăn). Phải tồn tại mê cung mà đường ít cú lăn nhất lại **dài hơn** đường nhiều cú lăn. Ta viết bản sai rồi tìm một mê cung ngẫu nhiên khác nhau với bản đúng:

In [22]:
def wrong_bfs_rolls(maze, start, destination):
    """SAI: BFS đếm số cú lăn, rồi lại trả về độ dài của đường tìm thấy đầu tiên."""
    m, n = len(maze), len(maze[0])
    seen = {tuple(start)}
    q = deque([(start[0], start[1], 0)])
    while q:
        r, c, total = q.popleft()
        if [r, c] == list(destination):
            return total
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, total + steps))
    return -1

rng = random.Random(7)
found = None
for _ in range(20000):
    mm, nn = rng.randint(3, 6), rng.randint(3, 6)
    g = [[1 if rng.random() < 0.25 else 0 for _ in range(nn)] for _ in range(mm)]
    empties = [(i, j) for i in range(mm) for j in range(nn) if g[i][j] == 0]
    if len(empties) < 2:
        continue
    s, d = rng.sample(empties, 2)
    right = shortest_dijkstra(g, list(s), list(d))
    wrong = wrong_bfs_rolls(g, list(s), list(d))
    if right != wrong:
        found = (g, list(s), list(d), right, wrong)
        break

g, s, d, right, wrong = found
show(g, s, d)
print("start", s, "dest", d, "| đúng:", right, "| BFS sai:", wrong)
assert right < wrong

. . . . D . 
# . . . . . 
. . . . . . 
. # S # . . 
# . . . . . 
. . # # . . 
start [3, 2] dest [0, 4] | đúng: 11 | BFS sai: 17


Bản BFS tính theo số cú lăn tìm thấy đường có ít cú lăn nhất, nhưng độ dài thật **lớn hơn** đường tối ưu. Đây là lý do cần Dijkstra/BFS-bước-đơn-vị.

**Sai 2 — dừng ngay khi đi ngang qua đích** (đếm đích như một ô bất kỳ trên đường lăn). Ví dụ 2 là phản ví dụ sẵn có: đáp án đúng `-1` nhưng bản sai sẽ trả về một số dương.

In [23]:
def wrong_pass_through(maze, start, destination):
    """SAI: coi mọi ô bóng đi qua là 'tới được', kể cả khi không dừng."""
    m, n = len(maze), len(maze[0])
    dist = {tuple(start): 0}
    heap = [(0, start[0], start[1])]
    best = math.inf
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist.get((r, c), math.inf):
            continue
        for dr, dc in DIRS:
            x, y, steps = r, c, 0
            while 0 <= x + dr < m and 0 <= y + dc < n and maze[x + dr][y + dc] == 0:
                x += dr
                y += dc
                steps += 1
                if [x, y] == list(destination):          # SAI: đi ngang qua cũng tính
                    best = min(best, d + steps)
            if steps and d + steps < dist.get((x, y), math.inf):
                dist[(x, y)] = d + steps
                heapq.heappush(heap, (d + steps, x, y))
    return -1 if best == math.inf else best

print("sai:", wrong_pass_through(maze1, [0, 4], [3, 2]), "| đúng:", shortest_dijkstra(maze1, [0, 4], [3, 2]))
assert wrong_pass_through(maze1, [0, 4], [3, 2]) != -1
assert shortest_dijkstra(maze1, [0, 4], [3, 2]) == -1

sai: 9 | đúng: -1


### 12.3. Đối chiếu ngẫu nhiên có seed

Ta sinh mê cung ngẫu nhiên đúng ràng buộc (ít nhất 2 ô trống, `start != destination`, cả hai là ô trống) và so sánh **cả 5 cách** (vét cạn chỉ trên mê cung ≤ `5 x 5` để khỏi nổ tổ hợp). Cách 5 (BFS theo `(ô, hướng)`) cũng đóng vai **oracle độc lập** vì nó không dùng "cạnh trọng số".

In [24]:
def random_maze(rng, mmax, nmax, p_wall):
    while True:
        mm, nn = rng.randint(1, mmax), rng.randint(1, nmax)
        g = [[1 if rng.random() < p_wall else 0 for _ in range(nn)] for _ in range(mm)]
        empties = [(i, j) for i in range(mm) for j in range(nn) if g[i][j] == 0]
        if len(empties) >= 2:
            s, d = rng.sample(empties, 2)
            return g, list(s), list(d)

rng = random.Random(2024)
count = 0
for _ in range(1500):
    g, s, d = random_maze(rng, 5, 5, rng.choice([0.1, 0.25, 0.4]))
    ref = shortest_bfs_dir(g, s, d)
    for name, fn in solvers.items():
        assert fn(g, s, d) == ref, (name, g, s, d)
    assert Solution().shortestDistance(g, s, d) == ref
    count += 1
print("đã đối chiếu", count, "mê cung nhỏ: tất cả 5 cách + code nộp khớp nhau")

big = 0
for _ in range(300):
    g, s, d = random_maze(rng, 12, 12, rng.choice([0.1, 0.2, 0.3]))
    ref = shortest_bfs_dir(g, s, d)
    for name in ("spfa", "dijkstra", "astar", "bfs_dir"):
        assert solvers[name](g, s, d) == ref, (name, g, s, d)
    big += 1
print("đã đối chiếu", big, "mê cung cỡ tới 12x12 (không dùng vét cạn)")

đã đối chiếu 1500 mê cung nhỏ: tất cả 5 cách + code nộp khớp nhau


đã đối chiếu 300 mê cung cỡ tới 12x12 (không dùng vét cạn)


In [25]:
# Mê cung lớn nhất theo ràng buộc: 100 x 100, tường ngẫu nhiên thưa -> kiểm tra tốc độ và tính nhất quán
import time
rng = random.Random(99)
g = [[1 if rng.random() < 0.15 else 0 for _ in range(100)] for _ in range(100)]
g[0][0] = 0
g[99][99] = 0
t0 = time.time()
a = shortest_dijkstra(g, [0, 0], [99, 99])
b = shortest_bfs_dir(g, [0, 0], [99, 99])
c = shortest_astar(g, [0, 0], [99, 99])
d_ = Solution().shortestDistance(g, [0, 0], [99, 99])
assert a == b == c == d_, (a, b, c, d_)
print("100x100:", a, "| thời gian", round(time.time() - t0, 2), "giây (đã chạy 4 cách)")

100x100: 270 | thời gian 0.03 giây (đã chạy 4 cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số cú lăn thay vì số ô

**Code sai:**

In [26]:
def buggy_count_rolls(maze, start, destination):
    seen = {tuple(start)}
    q = deque([(start[0], start[1], 0)])
    while q:
        r, c, rolls = q.popleft()
        if [r, c] == list(destination):
            return rolls                       # trả về SỐ CÚ LĂN
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, rolls + 1))  # cộng 1 mỗi cú lăn
    return -1

print("sai:", buggy_count_rolls(maze1, [0, 4], [4, 4]), "| đúng:", shortest_dijkstra(maze1, [0, 4], [4, 4]))

sai: 7 | đúng: 12


**Vì sao sai:** đề đo bằng **số ô**, không phải số cú lăn. Ví dụ 1 chỉ có `7` cú lăn nhưng quãng đường là `12`. Phải cộng `steps`, không phải `1`.

**Code đúng:** dùng `d + steps` làm trọng số (Dijkstra) — như ở mục 9.

### Lỗi 2: Đánh dấu "đã thăm" ngay lần đầu thấy ô (BFS thường với trọng số không đều)

In [27]:
def buggy_visited_first(maze, start, destination):
    seen = {tuple(start)}
    q = deque([(start[0], start[1], 0)])
    best = math.inf
    while q:
        r, c, total = q.popleft()
        if [r, c] == list(destination):
            best = min(best, total)
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and (nr, nc) not in seen:
                seen.add((nr, nc))            # ĐÁNH DẤU QUÁ SỚM: chưa chắc đường này ngắn nhất
                q.append((nr, nc, total + steps))
    return -1 if best == math.inf else best

# tìm một mê cung mà bản này sai
rng = random.Random(5)
for _ in range(20000):
    g, s, d = random_maze(rng, 6, 6, 0.25)
    if buggy_visited_first(g, s, d) != shortest_bfs_dir(g, s, d):
        break
show(g, s, d)
print("start", s, "dest", d, "| sai:", buggy_visited_first(g, s, d), "| đúng:", shortest_bfs_dir(g, s, d))

# # . . # . 
. . . D # # 
. . # . . . 
. . . . . S 
. # # . . # 
. # # . . . 
start [3, 5] dest [1, 3] | sai: 10 | đúng: 8


**Vì sao sai:** BFS chỉ đúng khi mọi cạnh có cùng trọng số. Ở đây cạnh nặng 1, 2, 3, ... nên ô được thấy **đầu tiên** chưa chắc được tới bằng đường **ngắn nhất**. Phải cho phép nới lỏng (cách 2) hoặc dùng heap (cách 3), hoặc đổi sang trạng thái bước đơn vị (cách 5).

### Lỗi 3: Lăn lố vào tường (kiểm tra sau khi bước)

In [28]:
def buggy_roll(maze, r, c, dr, dc):
    m, n = len(maze), len(maze[0])
    steps = 0
    while 0 <= r < m and 0 <= c < n and maze[r][c] == 0:   # bước trước, kiểm tra sau
        r += dr
        c += dc
        steps += 1
    return r, c, steps          # (r, c) đã nằm TRONG tường / ngoài biên

print("sai :", buggy_roll(maze1, 0, 4, 0, -1))
print("đúng:", roll(maze1, 0, 4, 0, -1))

sai : (0, 2, 2)
đúng: (0, 3, 1)


**Vì sao sai:** nếu bước rồi mới kiểm tra thì ô cuối cùng là **tường**, không phải ô dừng; số ô đếm cũng dư 1. Hoặc phải lùi lại một bước (`r -= dr; c -= dc; steps -= 1`). Cách gọn nhất: kiểm tra **ô kế tiếp** trước khi bước, như hàm `roll` ở mục 3.3.

### Lỗi 4: Coi đích là đã tới khi bóng đi ngang qua

Đã minh họa ở mục 12.2 (`wrong_pass_through`). Cách đúng: chỉ xét "đích" **sau khi cú lăn kết thúc**, tức là xét tại ô dừng `(nr, nc)`, không xét trong vòng `while`.

### Lỗi 5: `[[inf] * n] * m` làm các hàng dùng chung một list

Đã thấy ở mục 3.5. Phải dùng `[[inf] * n for _ in range(m)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "lăn/trượt cho tới khi chạm vật cản" → **đừng đi từng ô theo kiểu lưới thường**; đổi đỉnh thành **ô dừng** và cạnh thành **cú lăn**.
- Thấy "đường đi ngắn nhất" mà cạnh có **trọng số khác nhau, dương** → **Dijkstra** (heap). Trọng số đều (đều bằng 1) → BFS.
- Nếu muốn dùng BFS cho trọng số không đều, có thể **chẻ cạnh** thành các cạnh nặng 1 (bước đơn vị) và thêm thông tin vào trạng thái (ở đây là **hướng**).
- Luôn nhớ phần **điều kiện dừng ở đích**: "đi qua" khác "dừng tại".
- Mẫu code Dijkstra trên lưới: `dist` 2 chiều + heap `(d, r, c)` + `if d > dist[r][c]: continue`.
- Với heap, **bỏ qua bản cũ** (lazy deletion) thay vì tìm và xóa trong heap.
- Đừng quên các biến thể liên quan: *The Maze* (bài 490, chỉ hỏi có tới được không), *The Maze III* (bài 499, in cả đường đi).

## 15. Biến thể

1. **The Maze (490):** chỉ hỏi có thể dừng ở đích không → BFS/DFS trên đồ thị ô dừng, không cần khoảng cách.
2. **The Maze III (499):** có **lỗ** (hole); bóng rơi vào lỗ khi đi ngang qua nó. Trả về chuỗi hướng ngắn nhất (và nhỏ nhất theo thứ tự từ điển nếu hòa).
3. **Trả về cả đường đi:** lưu `parent[(r, c)] = (pr, pc, hướng)` khi nới lỏng, rồi truy ngược từ đích.
4. **Đếm số ô dừng khác nhau** có thể tới được từ `start`.
5. **Bóng có thể dừng giữa chừng tối đa `k` lần** → thêm `k` vào trạng thái.
6. **Có trọng số theo loại ô** (bùn mất 2, cỏ mất 1) → vẫn Dijkstra, trọng số cú lăn là tổng chi phí các ô.

Dưới đây là biến thể 3, trả về cả chuỗi hướng (ký tự `u`, `d`, `l`, `r`) và khoảng cách:

In [29]:
def shortest_with_path(maze, start, destination):
    """Trả về (khoảng cách, chuỗi hướng 'udlr'); (-1, '') nếu không tới được."""
    m, n = len(maze), len(maze[0])
    letters = "udlr"
    dist = {tuple(start): 0}
    parent = {}
    heap = [(0, start[0], start[1])]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist.get((r, c), math.inf):
            continue
        if [r, c] == list(destination):
            path = []
            cur = (r, c)
            while cur in parent:
                cur, letter = parent[cur]
                path.append(letter)
            return d, "".join(reversed(path))
        for k, (dr, dc) in enumerate(DIRS):
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps and d + steps < dist.get((nr, nc), math.inf):
                dist[(nr, nc)] = d + steps
                parent[(nr, nc)] = ((r, c), letters[k])
                heapq.heappush(heap, (d + steps, nr, nc))
    return -1, ""

print(shortest_with_path(maze1, [0, 4], [4, 4]))
print(shortest_with_path(maze1, [0, 4], [3, 2]))

(12, 'ldldrdr')
(-1, '')


## 16. So sánh các cách làm

| Cách | Ý tưởng chính | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn DFS | Liệt kê mọi đường đơn | cấp mũ | `O(V)` | Chỉ để kiểm chứng trên mê cung rất nhỏ |
| 2. SPFA | Hàng đợi, đưa lại vào hàng đợi khi `dist` giảm | `O(V·E·L)` xấu nhất | `O(mn)` | Khi muốn code ngắn, dữ liệu nhỏ |
| 3. Dijkstra | Heap, chốt ô có `dist` nhỏ nhất | `O(mn·L + mn·log mn)` | `O(mn)` | **Cách chuẩn khi thi/phỏng vấn** |
| 4. A* | Dijkstra + heuristic Manhattan | như Dijkstra | `O(mn)` | Đích xa, muốn mở rộng ít ô |
| 5. BFS `(ô, hướng)` | Bước đơn vị, cạnh nặng 1 | `O(mn)` | `O(mn)` | Khi muốn tránh heap, hằng số nhỏ |

Chọn cách 3 để nộp: ngắn, dễ chứng minh, độ phức tạp tốt.

## 17. 5 bài tập vận dụng (dễ → khó)

Mỗi bài có cell đề. Cell **TODO** chỉ định nghĩa hàm rỗng/gán `None`, **không được gọi** khi chạy notebook; hãy tự điền rồi chạy kiểm tra của riêng bạn. Đáp án ở mục 19.

**Bài 1 (đoán output).** Chạy trong đầu rồi so với máy: với `row = [[0, 0, 0, 1, 0]]`, `roll(row, 0, 0, 0, 1)` trả về gì?

In [30]:
row = [[0, 0, 0, 1, 0]]
# Dự đoán trước khi chạy: (hàng dừng, cột dừng, số ô) = ?
print(roll(row, 0, 0, 0, 1))

(0, 2, 2)


**Bài 2 (tính tay).** Cho mê cung `2 x 3` toàn ô trống `[[0,0,0],[0,0,0]]`, `start = [0,0]`, `destination = [1,2]`. Hãy tính tay quãng đường ngắn nhất, liệt kê dãy cú lăn. Ghi đáp án vào biến `ans2` (gán `None` để chờ bạn điền).

In [31]:
ans2 = None      # TODO: điền số nguyên

**Bài 3 (điền TODO).** Viết hàm `reachable_stops(maze, start)` trả về **tập** các ô dừng mà bóng có thể tới (kể cả `start`) bằng một chuỗi cú lăn nào đó. Dùng BFS/DFS trên đồ thị ô dừng và hàm `roll` đã có.

In [32]:
def reachable_stops(maze, start):
    # TODO: BFS từ start; dùng roll(maze, r, c, dr, dc) và tập visited
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết lại **Dijkstra** thành hàm `my_dijkstra(maze, start, destination)`. Gợi ý: `dist`, `heap`, bỏ qua bản cũ, `roll`.

In [33]:
def my_dijkstra(maze, start, destination):
    # TODO: tự viết lại từ đầu
    pass

**Bài 5 (mini challenge).** *The Maze (bài 490):* viết `can_stop(maze, start, destination)` trả về `True/False` — bóng có thể **dừng** ở đích không (không cần quãng đường). Sau đó nâng cấp: viết `count_stops(maze, start)` trả về **số** ô dừng khác nhau (không tính `start` nếu chỉ tới được bằng 0 cú lăn).

In [34]:
def can_stop(maze, start, destination):
    # TODO
    pass

def count_stops(maze, start):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: bóng đi tới ô ngay trước tường. Tường ở cột nào? Đếm số ô đã đi.
- Bài 2: thử "phải rồi xuống" và "xuống rồi phải".
- Bài 3: đồ thị ô dừng, dùng hàng đợi và tập `visited`.
- Bài 4: nhớ heap chứa `(d, r, c)`.
- Bài 5: bài 5a giống bài 3 rồi kiểm tra `destination in visited`.

**Mức 2 (vừa):**
- Bài 2: mỗi cú lăn đi sát tường; từ `(0,0)` phải tới `(0,2)` (2 ô) rồi xuống `(1,2)` (1 ô). Có đường nào ngắn hơn không?
- Bài 3: với mỗi ô lấy ra, gọi `roll` 4 hướng; chỉ thêm ô dừng nếu `steps > 0` và chưa thăm.
- Bài 4: nới lỏng khi `d + steps < dist[nr][nc]`; trả `-1` khi heap rỗng.
- Bài 5: `count_stops` = `len(reachable_stops(...)) - 1` hay không? Hãy nghĩ ô `start` có thể được tới lại bằng cú lăn khác.

**Mức 3 (gần đáp án):**
- Bài 3: `visited = {start}`; `q = deque([start])`; vòng `while q`.
- Bài 4: xem lại mục 9, thay `float('inf')` bằng `math.inf` cũng được.
- Bài 5: `can_stop` = `tuple(destination) in reachable_stops(maze, start)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert` cho từng bài.

In [35]:
# Bài 1: đi 2 ô, dừng ở cột 2 (cột 3 là tường)
assert roll(row, 0, 0, 0, 1) == (0, 2, 2)

# Bài 2: phải 2 ô rồi xuống 1 ô = 3; hoặc xuống 1 ô rồi phải 2 ô = 3. Không có đường nào ngắn hơn (Manhattan = 3).
ans2 = 3
assert ans2 == shortest_dijkstra([[0, 0, 0], [0, 0, 0]], [0, 0], [1, 2])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [36]:
# Bài 3
def reachable_stops(maze, start):
    s0 = (start[0], start[1])
    visited = {s0}
    q = deque([s0])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and (nr, nc) not in visited:
                visited.add((nr, nc))
                q.append((nr, nc))
    return visited

stops = reachable_stops(maze1, [0, 4])
assert (4, 4) in stops and (3, 2) not in stops
assert (0, 4) in stops
print(sorted(stops))

[(0, 0), (0, 1), (0, 3), (0, 4), (1, 0), (1, 2), (1, 3), (1, 4), (2, 0), (2, 1), (2, 2), (2, 4), (4, 0), (4, 2), (4, 4)]


In [37]:
# Bài 4
def my_dijkstra(maze, start, destination):
    m, n = len(maze), len(maze[0])
    dist = [[math.inf] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    heap = [(0, start[0], start[1])]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[r][c]:
            continue
        if [r, c] == list(destination):
            return d
        for dr, dc in DIRS:
            nr, nc, steps = roll(maze, r, c, dr, dc)
            if steps > 0 and d + steps < dist[nr][nc]:
                dist[nr][nc] = d + steps
                heapq.heappush(heap, (d + steps, nr, nc))
    return -1

for mz, s, d, exp in examples:
    assert my_dijkstra(mz, s, d) == exp
rng = random.Random(11)
for _ in range(300):
    g, s, d = random_maze(rng, 6, 6, 0.25)
    assert my_dijkstra(g, s, d) == shortest_bfs_dir(g, s, d)
print("Bài 4 OK")

Bài 4 OK


In [38]:
# Bài 5
def can_stop(maze, start, destination):
    return tuple(destination) in reachable_stops(maze, start)

def count_stops(maze, start):
    stops = reachable_stops(maze, start)
    # start luôn nằm trong tập (ta thêm sẵn); nó chỉ "thật sự" là điểm dừng nếu có cú lăn quay lại nó
    back = any(
        roll(maze, r, c, dr, dc)[:2] == (start[0], start[1]) and roll(maze, r, c, dr, dc)[2] > 0
        for (r, c) in stops for dr, dc in DIRS
    )
    return len(stops) - (0 if back else 1)

assert can_stop(maze1, [0, 4], [4, 4]) is True
assert can_stop(maze1, [0, 4], [3, 2]) is False
assert can_stop(maze3, [4, 3], [0, 1]) is False
for mz, s, d, exp in examples:
    assert can_stop(mz, s, d) == (exp != -1)
assert count_stops([[0, 0]], [0, 0]) == 2      # tới được (0,1); từ (0,1) lăn quay lại (0,0) nên (0,0) cũng là điểm dừng
print("count_stops ví dụ 1:", count_stops(maze1, [0, 4]))
print("Bài 5 OK")

count_stops ví dụ 1: 15
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Dijkstra cho mê cung "lăn tới tường":**

```python
def shortestDistance(self, maze, start, destination):
    m, n = len(maze), len(maze[0])
    dist = [[float('inf')] * n for _ in range(m)]
    dist[start[0]][start[1]] = 0
    heap = [(0, start[0], start[1])]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[r][c]:
            continue
        if [r, c] == destination:
            return d
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc, steps = r, c, 0
            while 0 <= nr + dr < m and 0 <= nc + dc < n and maze[nr + dr][nc + dc] == 0:
                nr += dr; nc += dc; steps += 1
            if steps and d + steps < dist[nr][nc]:
                dist[nr][nc] = d + steps
                heapq.heappush(heap, (d + steps, nr, nc))
    return -1
```

**Checklist trước khi submit:**

- [ ] Đỉnh = ô **dừng**; cạnh = **cú lăn** có trọng số = số ô?
- [ ] Kiểm tra ô **kế tiếp** trước khi bước (không lố vào tường)?
- [ ] Bỏ qua cú lăn `steps == 0`?
- [ ] Đích chỉ tính khi bóng **dừng** ở đó (xét sau vòng `while`)?
- [ ] Dùng `d + steps`, không phải `+ 1`?
- [ ] Heap: bỏ qua bản cũ `d > dist[r][c]`; trả `-1` khi heap cạn?
- [ ] `dist` tạo bằng list comprehension, không phải `[[inf]*n]*m`?
- [ ] Đã chạy ví dụ 1, 2, 3 và vài mê cung 1xN, Nx1?